# 01 — Dataset exploration

Partitions used: **train** and **student_test** only. The evaluator-testing partition is never opened (Dataset Exploration §7).

All numbers below are computed from the raw files by `src/data/exploration.py`; the same analysis runs headless with `python scripts/run_eda.py`.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, IFrame, display
from src.config import load_config, resolve
cfg = load_config()
RESULTS = resolve(cfg["paths"]["results_dir"])

In [ ]:
from src.data.io import list_partition
for p in ("train", "student_test"):
    print(p, len(list_partition(cfg, p)), "trips")

## Raw file inspection
Look at the first rows of each file type for one training trip to confirm column order and dtypes.

In [ ]:
from src.data.io import iter_partition
trip = next(iter_partition(cfg, "train"))
print(trip.trip_id, {k: v.name for k, v in trip.files.items()})
display(trip.track.head()); print(trip.track.dtypes)
print("nodes", trip.nodes.shape, trip.nodes[:3]); print("arcs", trip.arcs.shape, trip.arcs[:3])
print("route", None if trip.route is None else trip.route[:10])

## Inventory, validation, network connectivity

In [ ]:
from src.data.exploration import explore
tables = explore(cfg, RESULTS / "eda")
display(tables["partition_summary"])
display(tables["validation"].drop(columns=["issues"]).describe().T)

In [ ]:
v = tables["validation"]
display(v.loc[v["n_issues"] > 0, ["partition", "trip_id", "issues"]])

In [ ]:
display(tables["network"].describe().T)

## Plots

In [ ]:
for name in ["obs_per_trip", "sampling_interval", "trip_length_km", "gps_to_route_distance",
             "network_vs_track", "missing_and_anomalies"]:
    p = RESULTS / "eda" / f"{name}.png"
    if p.exists(): display(Image(filename=str(p)))

## Folium map of a training trip

In [ ]:
from src.data.prepare import build_network
from src.visualization.folium_maps import trip_map
net = build_network(trip.inputs(), cfg)
trip_map(net, trip.track[["lon", "lat"]].to_numpy(), trip.route, title=f"Training trip {trip.trip_id}")

## Label derivation check
Per-observation labels come from aligning the `.route` arc sequence to the GPS points (`src/data/labels.py`). The residual distribution shows how well the reference route explains the GPS.

In [ ]:
from src.data.prepare import prepare_labelled
prep, lab = prepare_labelled(trip, cfg)
print(pd.Series(lab.alignment.residual_m).describe())
print("candidate miss rate:", 1 - prep.features.groupby("obs")["label"].max().reindex(range(len(prep.track)), fill_value=0).mean())

## Data-quality issues, assumptions and decisions

*Fill in from the outputs above (team-written).*